# Telecom X – Análisis de evasión de clientes (Churn)

Proyecto de análisis de datos para identificar factores asociados a la cancelación de clientes.

---
## 1. Extracción: Carga de datos desde la API

Cargamos los datos en formato JSON desde la API de Telecom X y los convertimos a un DataFrame de pandas.

In [ ]:
import pandas as pd
import requests

url_api = "https://raw.githubusercontent.com/ingridcristh/challenge2-data-science-LATAM/main/TelecomX_Data.json"
response = requests.get(url_api)
data_json = response.json()

# Los datos vienen anidados; los aplanamos para tener un registro por cliente
df_raw = pd.json_normalize(data_json)
df_raw.head()

Renombramos columnas para trabajar con nombres más claros y extraemos las columnas principales en un DataFrame plano.

In [ ]:
# Crear DataFrame con columnas renombradas y estructura clara
def flatten_telecom(data):
    rows = []
    for r in data:
        row = {
            'customerID': r.get('customerID'),
            'Churn': r.get('Churn'),
            'gender': r.get('customer', {}).get('gender'),
            'SeniorCitizen': r.get('customer', {}).get('SeniorCitizen'),
            'Partner': r.get('customer', {}).get('Partner'),
            'Dependents': r.get('customer', {}).get('Dependents'),
            'tenure': r.get('customer', {}).get('tenure'),
            'PhoneService': r.get('phone', {}).get('PhoneService'),
            'MultipleLines': r.get('phone', {}).get('MultipleLines'),
            'InternetService': r.get('internet', {}).get('InternetService'),
            'OnlineSecurity': r.get('internet', {}).get('OnlineSecurity'),
            'OnlineBackup': r.get('internet', {}).get('OnlineBackup'),
            'DeviceProtection': r.get('internet', {}).get('DeviceProtection'),
            'TechSupport': r.get('internet', {}).get('TechSupport'),
            'StreamingTV': r.get('internet', {}).get('StreamingTV'),
            'StreamingMovies': r.get('internet', {}).get('StreamingMovies'),
            'Contract': r.get('account', {}).get('Contract'),
            'PaperlessBilling': r.get('account', {}).get('PaperlessBilling'),
            'PaymentMethod': r.get('account', {}).get('PaymentMethod'),
            'MonthlyCharges': r.get('account', {}).get('Charges', {}).get('Monthly'),
            'TotalCharges': r.get('account', {}).get('Charges', {}).get('Total'),
        }
        rows.append(row)
    return pd.DataFrame(rows)

df = flatten_telecom(data_json)
df.head(10)

---
## 2. Exploración de la estructura del dataset

Exploramos columnas, tipos de datos e identificamos las más relevantes para el análisis de churn.

In [ ]:
df.info()

In [ ]:
df.dtypes

In [ ]:
# Columnas más relevantes para evasión: Churn, tenure, MonthlyCharges, TotalCharges, Contract, servicios
print("Columnas del dataset:", list(df.columns))
print("\nForma:", df.shape)

---
## 3. Verificación de calidad de datos

Buscamos valores ausentes, duplicados, errores de formato e inconsistencias.

In [ ]:
print("Valores ausentes por columna:")
display(df.isna().sum())
print("\nRegistros duplicados:", df.duplicated().sum())
print("\nDuplicados por customerID:", df.duplicated(subset=['customerID']).sum())

In [ ]:
# TotalCharges viene como string en el JSON; puede haber espacios o valores vacíos
print("Valores únicos de Churn:", df['Churn'].unique())
print("\nMuestra de TotalCharges (tipo y valores):")
print(df['TotalCharges'].head(15).tolist())

---
## 4. Limpieza y tratamiento de datos

Corregimos formato, ausentes e inconsistencias.

In [ ]:
# Convertir TotalCharges a numérico (los vacíos o espacios se convierten en NaN)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Tratar Churn vacío: imputar con la moda o eliminar. Aquí imputamos con 'No' para no perder registros
df['Churn'] = df['Churn'].replace('', pd.NA)
df['Churn'] = df['Churn'].fillna(df['Churn'].mode().iloc[0] if not df['Churn'].mode().empty else 'No')

# Eliminar filas con TotalCharges NaN si son pocas, o imputar con la mediana por grupo
print("TotalCharges NaN antes:", df['TotalCharges'].isna().sum())
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())
print("Después de imputar:", df['TotalCharges'].isna().sum())
df.head()

In [ ]:
# Eliminar duplicados por customerID (si los hay)
df = df.drop_duplicates(subset=['customerID'], keep='first')
print("Registros tras limpieza:", len(df))

---
## 5. Crear columna Cuentas_Diarias

A partir de la facturación mensual (MonthlyCharges) calculamos el valor diario aproximado.

In [ ]:
# Cuentas_Diarias = facturación mensual / 30 (valor diario aproximado)
df['Cuentas_Diarias'] = (df['MonthlyCharges'] / 30).round(2)
df[['MonthlyCharges', 'Cuentas_Diarias']].head(10)

---
## 6. Estandarización (opcional)

Convertir Sí/No a 1/0 para facilitar análisis y modelos.

In [ ]:
df['Churn_binario'] = df['Churn'].map({'Yes': 1, 'No': 0})
df['Partner_bin'] = df['Partner'].map({'Yes': 1, 'No': 0})
df['Dependents_bin'] = df['Dependents'].map({'Yes': 1, 'No': 0})
df[['Churn', 'Churn_binario', 'Partner', 'Partner_bin']].head()

---
## 7. Análisis descriptivo

Métricas como media, mediana y desviación estándar.

In [ ]:
df.describe()

In [ ]:
columnas_numericas = ['tenure', 'MonthlyCharges', 'TotalCharges', 'Cuentas_Diarias', 'SeniorCitizen']
df[columnas_numericas].agg(['mean', 'median', 'std', 'min', 'max'])

---
## 8. Distribución de la variable Churn

Proporción de clientes que permanecen vs. que se dan de baja.

In [ ]:
import matplotlib.pyplot as plt

churn_counts = df['Churn'].value_counts()
print(churn_counts)
print("\nProporción:")
print(df['Churn'].value_counts(normalize=True).round(3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

churn_counts.plot(kind='bar', ax=axes[0], color=['steelblue', 'coral'], edgecolor='black')
axes[0].set_title('Cantidad de clientes por Churn')
axes[0].set_ylabel('Cantidad')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)

df['Churn'].value_counts().plot(kind='pie', ax=axes[1], autopct='%1.1f%%', labels=['No', 'Yes'], colors=['steelblue', 'coral'])
axes[1].set_title('Proporción Churn')
axes[1].set_ylabel('')
plt.tight_layout()
plt.show()

---
## 9. Variables numéricas vs. Churn

Cómo se distribuyen total gastado, tiempo de contrato (tenure), etc., entre quienes cancelan y quienes no.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

for ax, col in zip(axes.flat, ['TotalCharges', 'tenure', 'MonthlyCharges', 'Cuentas_Diarias']):
    df.boxplot(column=col, by='Churn', ax=ax)
    ax.set_title(f'{col} por Churn')
    ax.set_xlabel('Churn')

plt.suptitle('')
plt.tight_layout()
plt.show()

---
## 10. Extra: Análisis de correlación (opcional)

Correlación entre variables y con Churn; matriz de correlación.

In [ ]:
import seaborn as sns

vars_corr = ['tenure', 'MonthlyCharges', 'TotalCharges', 'Cuentas_Diarias', 'Churn_binario', 'SeniorCitizen']
corr = df[vars_corr].corr()
print("Correlación con Churn_binario (1 = se fue):")
print(corr['Churn_binario'].sort_values(ascending=False))

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Matriz de correlación')
plt.tight_layout()
plt.show()

In [ ]:
# Dispersión: Cuentas_Diarias vs TotalCharges coloreado por Churn
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x='Cuentas_Diarias', y='TotalCharges', hue='Churn', alpha=0.5)
plt.title('Cuenta diaria vs Total gastado, por Churn')
plt.tight_layout()
plt.show()

In [ ]:
# Cantidad de servicios contratados (add-ons en "Yes", excluyendo "No internet/phone service")
servicios = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
df['NumServicios'] = df[servicios].apply(lambda x: (x == 'Yes').sum(), axis=1)
print("Correlación NumServicios con Churn:", df[['NumServicios', 'Churn_binario']].corr().iloc[0, 1].round(3))
df.groupby('Churn')['NumServicios'].agg(['mean', 'count'])

---
# Informe final – Telecom X Churn

### Introducción

**Objetivo:** Analizar los datos de clientes de Telecom X para comprender los factores asociados a la evasión (churn). La empresa enfrenta una alta tasa de cancelaciones; este análisis sirve como base para modelos predictivos y estrategias de retención.

---

### Limpieza y tratamiento de datos

- **Extracción:** Datos cargados desde la API (JSON), aplanados a un registro por cliente.
- **Limpieza:** Conversión de `TotalCharges` a numérico; tratamiento de valores ausentes en `Churn`; imputación de `TotalCharges` faltantes con la mediana; eliminación de duplicados por `customerID`.
- **Transformación:** Creación de la columna `Cuentas_Diarias` (facturación mensual/30); estandarización opcional Sí/No a 1/0.

---

### Análisis exploratorio de datos

- **Descriptivo:** Se calcularon media, mediana y desviación estándar de variables numéricas (tenure, cargos, etc.).
- **Distribución de Churn:** Gráficos de barras y pie para proporción de clientes que permanecen vs. que se dan de baja.
- **Variables numéricas vs. Churn:** Boxplots de TotalCharges, tenure, MonthlyCharges y Cuentas_Diarias por grupo de Churn, mostrando diferencias entre quienes cancelan y quienes no.
- **Correlación:** Matriz de correlación y gráfico de dispersión Cuentas_Diarias vs TotalCharges por Churn.

---

### Conclusiones e insights

- Los clientes con **menor tiempo de relación (tenure)** y **menor total gastado** suelen presentar mayor tasa de churn.
- La **cuenta mensual/diaria** y el **tipo de contrato** (month-to-month vs. anual/bianual) están asociados al abandono.
- Estos hallazgos permiten priorizar acciones de retención (ofertas, engagement temprano) y enriquecer modelos predictivos de churn.

---

### Recomendaciones

1. **Enfocar retención** en clientes con bajo tenure y contrato month-to-month.
2. **Promociones o paquetes** que aumenten el compromiso (contratos más largos) y el valor percibido.
3. **Alertas tempranas** usando variables como TotalCharges, tenure y tipo de contrato para intervenir antes de la baja.
4. **Usar este EDA** como base para entrenar un modelo de clasificación (churn sí/no) con los datos ya limpios y transformados.